# Phase 6 — Budget Impact: Exploration Notebook

This notebook ties everything together:
- Full pipeline from data to budget decision
- Compare budget outcomes under different distributional assumptions
- Visualize the practical consequences of model choice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats

from src.data_gen import generate_team_data, generate_salary_data, generate_severance_data
from src.fitting import fit_all
from src.model_selection import compare_models
from src.budget_impact import (
    var_at_level, expected_shortfall, budget_reserve,
    compare_distributions_impact,
)

sns.set_theme(style="whitegrid", font_scale=1.1)
SEED = 42

## 1. Generate Full Team Data

In [ ]:
team = generate_team_data(team_size=50, seed=SEED)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (name, values) in zip(axes.flat, team.items()):
    ax.hist(values, bins=20, alpha=0.7, color="steelblue", edgecolor="white")
    ax.set_title(f"{name.capitalize()} (n={len(values)}, mean=R${values.mean():,.0f})")
    ax.set_xlabel("R$")

plt.tight_layout()
plt.show()

## 2. Fit and Compare — Salary

In [ ]:
salary_fits = fit_all(team["salary"])
salary_table = compare_models(salary_fits)
salary_table[["Distribution", "AIC", "BIC", "AIC_weight", "delta_AIC"]]

## 3. Budget Impact — Salary

In [ ]:
impact = compare_distributions_impact(salary_fits, n_simulations=100_000, seed=SEED)
impact[["Distribution", "Mean", "Std", "VaR_95%", "ES_95%", "Reserve_95%", "VaR_99%", "ES_99%"]]

## 4. The Wrong Distribution Scenario

In [ ]:
# Generate large salary sample for clear demonstration
salary_large = generate_salary_data(2000, seed=SEED)
fits_large = fit_all(salary_large, candidates=["normal", "lognormal"])

rng = np.random.default_rng(SEED)

# Simulate from each
ln_fit = [f for f in fits_large if f.distribution == "LogNormal"][0]
n_fit = [f for f in fits_large if f.distribution == "Normal"][0]

sim_ln = rng.lognormal(ln_fit.params["mu"], ln_fit.params["sigma"], size=500_000)
sim_n = rng.normal(n_fit.params["mu"], n_fit.params["sigma"], size=500_000)

# Compare at different budget ceilings
mean_sal = salary_large.mean()
multiples = np.linspace(1.0, 3.0, 20)
p_over_ln = [(sim_ln > mean_sal * m).mean() for m in multiples]
p_over_n = [(sim_n > mean_sal * m).mean() for m in multiples]

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(multiples, p_over_ln, "o-", linewidth=2, color="#2A9D8F", label="LogNormal (correct)")
ax.semilogy(multiples, p_over_n, "s--", linewidth=2, color="#E63946", label="Normal (wrong)")
ax.set_xlabel("Budget ceiling (× mean salary)")
ax.set_ylabel("P(individual salary > ceiling) [log scale]")
ax.set_title("Overbudget Risk: Correct vs Wrong Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Total Budget Distribution (Monte Carlo Preview)

In [ ]:
# Simulate total annual salary cost for a 50-person team
# Under different distributional assumptions
n_sims = 50_000
team_size = 50

total_ln = np.array([
    rng.lognormal(ln_fit.params["mu"], ln_fit.params["sigma"], size=team_size).sum()
    for _ in range(n_sims)
])
total_n = np.array([
    rng.normal(n_fit.params["mu"], n_fit.params["sigma"], size=team_size).sum()
    for _ in range(n_sims)
])

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(total_ln / 1e6, bins=50, density=True, alpha=0.5, color="#2A9D8F", label="LogNormal model")
ax.hist(total_n / 1e6, bins=50, density=True, alpha=0.5, color="#E63946", label="Normal model")
ax.axvline(total_ln.mean() / 1e6, color="#2A9D8F", linestyle="--")
ax.axvline(total_n.mean() / 1e6, color="#E63946", linestyle="--")
ax.set_xlabel("Total Annual Salary Cost (R$ millions)")
ax.set_ylabel("Density")
ax.set_title("Total Budget Distribution: 50-Person Team")
ax.legend()
plt.tight_layout()
plt.show()

print(f"LogNormal model: mean=R${total_ln.mean()/1e6:.2f}M, VaR95=R${np.quantile(total_ln, 0.95)/1e6:.2f}M")
print(f"Normal model:    mean=R${total_n.mean()/1e6:.2f}M, VaR95=R${np.quantile(total_n, 0.95)/1e6:.2f}M")
print(f"Reserve difference (95%): R${(np.quantile(total_ln, 0.95) - np.quantile(total_n, 0.95))/1e3:,.0f}K")